# Modul 08: Klassifikation, Clustering und PCA mit NumPy

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Klassifikation mit NumPy, Cluster und PCA  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Grundlagen  
    **Orientierungszeit:** etwa 120 bis 165 Minuten

    ## Überblick

    Sie implementieren logistische Klassifikation, zentrale KMeans-Schritte und PCA mit NumPy. Die Aufgaben verbinden Wahrscheinlichkeiten, Kreuzentropie, Gradienten, Distanzen, Inertia, Eigenvektoren und erklärte Varianz.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_08A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_08B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Klassifikationsscores, Wahrscheinlichkeiten und binäre Verluste berechnen.
- Eine kleine logistische Regression mit NumPy trainieren.
- Logistische Regression, k-NN, Naive Bayes und Baseline fair vergleichen.
- KMeans-Schritte mit Distanzen, Zuordnung und Zentrenaktualisierung implementieren.
- Hauptkomponenten, erklärte Varianz und Projektionen mit NumPy berechnen.
- Clusterqualität, Gegenbeispiele und einfache Anomaliehinweise analysieren.

    ## Bewertete Fähigkeiten

    - stabile Sigmoid- und Kreuzentropieberechnung
- Gradienten logistischer Regression und Entscheidungsgrenze
- euklidische Distanzen, Zentrenupdate und Inertia
- Kovarianzmatrix, Eigenzerlegung, Projektion und erklärte Varianz

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_classification
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_binary, y_binary = make_classification(
    n_samples=260,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.25,
    flip_y=0.03,
    random_state=RANDOM_SEED,
)
X_clusters, hidden_cluster_labels = make_blobs(
    n_samples=180,
    centers=[(-4, -2), (0, 4), (4, -1)],
    cluster_std=[0.8, 1.0, 0.7],
    random_state=RANDOM_SEED,
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Logits, Sigmoid und Kreuzentropie stabil berechnen

    Implementieren Sie:

1. eine numerisch stabile Sigmoid-Funktion,
2. eine binäre Kreuzentropie für Wahrscheinlichkeiten,
3. Vorhersagelabels bei Schwellenwert 0.5.

Testen Sie die Funktionen mit Logits `[-1000, -2, 0, 2, 1000]` und passenden Labels. Vergleichen Sie die Verluste einer guten und einer absichtlich schlechten Wahrscheinlichkeitsvorhersage.

> **Hinweis:** Clipping gehört in den Logarithmus, nicht als Ersatz für die Sigmoid-Funktion.

In [ ]:
logits = np.array([-1000.0, -2.0, 0.0, 2.0, 1000.0])
labels = np.array([0, 0, 1, 1, 1], dtype=float)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Logits, Sigmoid und Kreuzentropie stabil berechnen
#
# Ziel dieser Codezelle:
# Implementieren Sie: 1. eine numerisch stabile Sigmoid-Funktion, 2. eine binäre
# Kreuzentropie für Wahrscheinlichkeiten, 3. Vorhersagelabels bei Schwellenwert 0.5.
# Testen Sie die Funktionen mit Logits [-1000, -2, 0, 2,...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

logits = np.array([-1000.0, -2.0, 0.0, 2.0, 1000.0])
labels = np.array([0, 0, 1, 1, 1], dtype=float)

def stable_sigmoid(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    output = np.empty_like(values)

    # Für nichtnegative Werte ist exp(-x) sicher. Für negative Werte
    # wird eine algebraisch äquivalente Form mit exp(x) verwendet.
    nonnegative = values >= 0
    output[nonnegative] = 1.0 / (1.0 + np.exp(-values[nonnegative]))
    exp_values = np.exp(values[~nonnegative])
    output[~nonnegative] = exp_values / (1.0 + exp_values)
    return output

def binary_cross_entropy(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> float:
    # Clipping verhindert log(0), ohne die Bedeutung der Werte sichtbar
    # zu verändern.
    epsilon = 1e-12
    probabilities = np.clip(probabilities, epsilon, 1.0 - epsilon)
    losses = -(
        y_true * np.log(probabilities)
        + (1.0 - y_true) * np.log(1.0 - probabilities)
    )
    return float(np.mean(losses))

probabilities = stable_sigmoid(logits)
predictions = (probabilities >= 0.5).astype(int)

good_probabilities = np.array([0.03, 0.15, 0.72, 0.84, 0.97])
bad_probabilities = 1.0 - good_probabilities

print("Wahrscheinlichkeiten:", probabilities)
print("Labels:", predictions)
print("Verlust guter Wahrscheinlichkeiten:", round(binary_cross_entropy(labels, good_probabilities), 4))
print("Verlust schlechter Wahrscheinlichkeiten:", round(binary_cross_entropy(labels, bad_probabilities), 4))

### Reflexion zu Aufgabe 1

Die Sigmoid-Funktion bildet beliebige Logits auf Werte zwischen 0 und 1 ab. Extreme Logits können bei einer naiven Implementierung Überlauf erzeugen. Kreuzentropie bestraft selbstsichere falsche Wahrscheinlichkeiten besonders stark. Der Schwellenwert 0,5 erzeugt Labels, ist aber von der probabilistischen Verlustberechnung getrennt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Logistische Regression mit NumPy trainieren

    Teilen Sie `X_binary`, `y_binary` stratifiziert in Training und Test und skalieren Sie mit Trainingsstatistiken. Implementieren Sie eine logistische Regression mit Bias:

- Vorwärtsrechnung,
- Kreuzentropie,
- Gradienten für Gewichte und Bias,
- 1.500 Gradientenschritte.

Zeichnen Sie die Verlustkurve und berechnen Sie die Testgenauigkeit.

> **Hinweis:** Achten Sie darauf, den Bias-Gradienten als Mittelwert der Fehler zu berechnen.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Logistische Regression mit NumPy trainieren
#
# Ziel dieser Codezelle:
# Teilen Sie Xbinary, ybinary stratifiziert in Training und Test und skalieren Sie
# mit Trainingsstatistiken. Implementieren Sie eine logistische Regression mit Bias:
# - Vorwärtsrechnung, - Kreuzentropie, - Gradienten für...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_binary,
    y_binary,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_binary,
)

# Skalierungsparameter werden ausschließlich aus dem Training gelernt.
feature_mean = X_train.mean(axis=0)
feature_std = X_train.std(axis=0, ddof=0)
X_train_scaled = (X_train - feature_mean) / feature_std
X_test_scaled = (X_test - feature_mean) / feature_std

def sigmoid(values: np.ndarray) -> np.ndarray:
    values = np.clip(values, -500, 500)
    return 1.0 / (1.0 + np.exp(-values))

weights = np.zeros(X_train_scaled.shape[1], dtype=float)
bias = 0.0
learning_rate = 0.10
epochs = 1500
losses = []

for epoch in range(epochs):
    logits_train = X_train_scaled @ weights + bias
    probabilities_train = sigmoid(logits_train)

    clipped = np.clip(probabilities_train, 1e-12, 1 - 1e-12)
    loss = -np.mean(
        y_train * np.log(clipped)
        + (1 - y_train) * np.log(1 - clipped)
    )
    losses.append(float(loss))

    # Für logistische Regression mit Kreuzentropie vereinfacht sich
    # der Gradient zu X.T @ (p - y) / n.
    errors = probabilities_train - y_train
    gradient_weights = X_train_scaled.T @ errors / len(y_train)
    gradient_bias = float(np.mean(errors))

    weights -= learning_rate * gradient_weights
    bias -= learning_rate * gradient_bias

test_probabilities = sigmoid(X_test_scaled @ weights + bias)
test_predictions = (test_probabilities >= 0.5).astype(int)
test_accuracy = accuracy_score(y_test, test_predictions)

print("Gewichte:", np.round(weights, 3))
print("Bias:", round(bias, 3))
print("Testgenauigkeit:", round(test_accuracy, 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses)
ax.set_title("Kreuzentropie während des Trainings")
ax.set_xlabel("Epoche")
ax.set_ylabel("Verlust")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 2

Das Modell lernt eine lineare Entscheidungsgrenze im skalierten Merkmalsraum. Die Verlustkurve sollte anfangs deutlich sinken und sich anschließend abflachen. Eine sinkende Trainingskreuzentropie garantiert noch keine gute Generalisierung, daher wird die Genauigkeit auf zuvor nicht verwendeten Testdaten berechnet.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Entscheidungsgrenze und klassische Baselines vergleichen

    Verwenden Sie denselben Train/Test-Split aus Aufgabe 2.

1. Trainieren Sie `DummyClassifier`, `KNeighborsClassifier(k=5)` und `GaussianNB`.
2. Verwenden Sie für k-NN dieselben skalierten Merkmale.
3. Vergleichen Sie die Testgenauigkeiten mit Ihrer NumPy-logistischen Regression.
4. Visualisieren Sie die Entscheidungsgrenze des NumPy-Modells zusammen mit den Testpunkten.

> **Hinweis:** Die skalierte Darstellung ist nicht in den ursprünglichen Maßeinheiten.

In [ ]:
# Führen Sie Aufgabe 2 zuerst aus, damit Split, Skalierung und NumPy-Modell verfügbar sind.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Entscheidungsgrenze und klassische Baselines vergleichen
#
# Ziel dieser Codezelle:
# Verwenden Sie denselben Train/Test-Split aus Aufgabe 2. 1. Trainieren Sie
# DummyClassifier, KNeighborsClassifier(k=5) und GaussianNB. 2. Verwenden Sie für
# k-NN dieselben skalierten Merkmale. 3. Vergleichen Sie die Test...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Die Dummy-Baseline nutzt nur die häufigste Trainingsklasse.
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_scaled, y_train)

# k-NN hängt stark von Distanzen ab und benötigt deshalb die Skalierung.
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# GaussianNB modelliert pro Klasse und Merkmal eine Gaußverteilung.
naive_bayes = GaussianNB()
naive_bayes.fit(X_train_scaled, y_train)

comparison = pd.DataFrame(
    {
        "model": ["Dummy", "k-NN", "GaussianNB", "NumPy-LogReg"],
        "accuracy": [
            accuracy_score(y_test, dummy.predict(X_test_scaled)),
            accuracy_score(y_test, knn.predict(X_test_scaled)),
            accuracy_score(y_test, naive_bayes.predict(X_test_scaled)),
            test_accuracy,
        ],
    }
).sort_values("accuracy", ascending=False)

print(comparison.round(3).to_string(index=False))

# Ein Gitter im skalierten Merkmalsraum wird durch das NumPy-Modell
# klassifiziert. Die Kontur bei Wahrscheinlichkeit 0,5 ist die Grenze.
x0_min, x0_max = X_test_scaled[:, 0].min() - 1, X_test_scaled[:, 0].max() + 1
x1_min, x1_max = X_test_scaled[:, 1].min() - 1, X_test_scaled[:, 1].max() + 1
grid_x0, grid_x1 = np.meshgrid(
    np.linspace(x0_min, x0_max, 220),
    np.linspace(x1_min, x1_max, 220),
)
grid_points = np.column_stack([grid_x0.ravel(), grid_x1.ravel()])
grid_probabilities = sigmoid(grid_points @ weights + bias).reshape(grid_x0.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(grid_x0, grid_x1, grid_probabilities, levels=[0.5], linewidths=2)
ax.scatter(
    X_test_scaled[:, 0],
    X_test_scaled[:, 1],
    c=y_test,
    edgecolor="black",
    alpha=0.8,
)
ax.set_title("Lineare Entscheidungsgrenze des NumPy-Modells")
ax.set_xlabel("Skaliertes Merkmal 1")
ax.set_ylabel("Skaliertes Merkmal 2")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 3

Die Dummy-Baseline prüft, ob ein Modell überhaupt mehr als die Klassenverteilung nutzt. k-NN kann gekrümmte lokale Grenzen erzeugen, GaussianNB beruht auf Verteilungsannahmen und logistische Regression erzeugt eine lineare Grenze. Der faire Vergleich verwendet denselben Split und dieselbe zulässige Vorverarbeitung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: KMeans-Schritte und Inertia mit NumPy implementieren

    Implementieren Sie KMeans für `X_clusters` mit drei Clustern.

1. Initialisieren Sie drei Zentren reproduzierbar aus vorhandenen Punkten.
2. Berechnen Sie alle quadrierten euklidischen Distanzen.
3. Ordnen Sie jeden Punkt dem nächsten Zentrum zu.
4. Aktualisieren Sie Zentren als Mittelwert ihrer Punkte.
5. Wiederholen Sie bis maximal 50 Iterationen oder bis sich die Zentren kaum ändern.
6. Speichern Sie die Inertia je Iteration und markieren Sie die fünf am weitesten von ihrem Zentrum entfernten Punkte als Anomaliehinweise.

> **Hinweis:** Berechnen Sie nach dem letzten Zentrenupdate Zuordnung und Distanzen erneut.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: KMeans-Schritte und Inertia mit NumPy implementieren
#
# Ziel dieser Codezelle:
# Implementieren Sie KMeans für Xclusters mit drei Clustern. 1. Initialisieren Sie
# drei Zentren reproduzierbar aus vorhandenen Punkten. 2. Berechnen Sie alle
# quadrierten euklidischen Distanzen. 3. Ordnen Sie jeden Punkt...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

k = 3
kmeans_rng = np.random.default_rng(RANDOM_SEED)
initial_indices = kmeans_rng.choice(len(X_clusters), size=k, replace=False)
centers = X_clusters[initial_indices].copy()
inertia_history = []

for iteration in range(50):
    # Broadcasting erzeugt für jeden Punkt und jedes Zentrum die
    # zweidimensionale Differenz. Summe über axis=2 ergibt Distanz².
    squared_distances = np.sum(
        (X_clusters[:, None, :] - centers[None, :, :]) ** 2,
        axis=2,
    )
    assignments = np.argmin(squared_distances, axis=1)
    nearest_squared_distance = squared_distances[
        np.arange(len(X_clusters)), assignments
    ]
    inertia_history.append(float(nearest_squared_distance.sum()))

    new_centers = centers.copy()
    for cluster_index in range(k):
        cluster_points = X_clusters[assignments == cluster_index]
        if len(cluster_points) > 0:
            new_centers[cluster_index] = cluster_points.mean(axis=0)

    center_shift = np.linalg.norm(new_centers - centers)
    centers = new_centers
    if center_shift < 1e-6:
        break

# Nach dem letzten Update werden Distanzen und Zuordnungen noch einmal
# konsistent zu den finalen Zentren berechnet.
final_squared_distances = np.sum(
    (X_clusters[:, None, :] - centers[None, :, :]) ** 2,
    axis=2,
)
assignments = np.argmin(final_squared_distances, axis=1)
distance_to_center = np.sqrt(
    final_squared_distances[np.arange(len(X_clusters)), assignments]
)
anomaly_indices = np.argsort(distance_to_center)[-5:]

print("Iterationen:", len(inertia_history))
print("Finale Inertia:", round(inertia_history[-1], 3))
print("Zentren:\n", np.round(centers, 3))
print("Anomaliehinweis-Indizes:", anomaly_indices)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_clusters[:, 0], X_clusters[:, 1], c=assignments, alpha=0.7)
ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=180, label="Zentren")
ax.scatter(
    X_clusters[anomaly_indices, 0],
    X_clusters[anomaly_indices, 1],
    facecolors="none",
    edgecolors="black",
    s=150,
    label="weit vom Zentrum",
)
ax.set_title("KMeans mit NumPy")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(inertia_history, marker="o")
ax.set_title("Inertia je KMeans-Iteration")
ax.set_xlabel("Iteration")
ax.set_ylabel("Summe der quadrierten Distanzen")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 4

KMeans minimiert die Summe quadrierter Abstände zu Zentren. Die Inertia sollte während der Iterationen nicht steigen. Das Verfahren bevorzugt ungefähr kompakte, ähnlich skalierte Gruppen und kann bei ungünstiger Initialisierung ein lokales Minimum erreichen. Große Zentrumabstände sind nur Anomaliehinweise, weil Randpunkte legitime Beobachtungen sein können.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: PCA mit NumPy und Clusterprojektion

    Erweitern Sie `X_clusters` um drei korrelierte Zusatzmerkmale, sodass ein fünfdimensionaler Datensatz entsteht. Implementieren Sie PCA mit NumPy:

1. Standardisieren Sie alle Merkmale.
2. Berechnen Sie Kovarianzmatrix, Eigenwerte und Eigenvektoren.
3. Sortieren Sie absteigend nach Eigenwert.
4. Berechnen Sie erklärte Varianzanteile und Projektion auf zwei Komponenten.
5. Visualisieren Sie die Projektion mit den KMeans-Zuordnungen aus Aufgabe 4.
6. Rekonstruieren Sie die standardisierten Daten aus zwei Komponenten und berechnen Sie den mittleren Rekonstruktionsfehler.

> **Hinweis:** Sortieren Sie Eigenwerte und die zugehörigen Eigenvektoren immer gemeinsam.

In [ ]:
# Führen Sie Aufgabe 4 zuerst aus, damit `assignments` verfügbar ist.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: PCA mit NumPy und Clusterprojektion
#
# Ziel dieser Codezelle:
# Erweitern Sie Xclusters um drei korrelierte Zusatzmerkmale, sodass ein
# fünfdimensionaler Datensatz entsteht. Implementieren Sie PCA mit NumPy: 1.
# Standardisieren Sie alle Merkmale. 2. Berechnen Sie Kovarianzmatrix, Ei...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Die Zusatzmerkmale sind verrauschte lineare Kombinationen der beiden
# Ausgangsmerkmale und erzeugen bewusst Redundanz.
X_five = np.column_stack(
    [
        X_clusters,
        0.7 * X_clusters[:, 0] + 0.2 * X_clusters[:, 1] + rng.normal(0, 0.25, len(X_clusters)),
        -0.3 * X_clusters[:, 0] + 0.9 * X_clusters[:, 1] + rng.normal(0, 0.25, len(X_clusters)),
        X_clusters[:, 0] + X_clusters[:, 1] + rng.normal(0, 0.35, len(X_clusters)),
    ]
)

feature_mean = X_five.mean(axis=0)
feature_std = X_five.std(axis=0, ddof=0)
X_standardized = (X_five - feature_mean) / feature_std

# rowvar=False bedeutet, dass Spalten Merkmale und Zeilen Beobachtungen sind.
covariance_matrix = np.cov(X_standardized, rowvar=False)

# eigh ist für symmetrische Matrizen wie Kovarianzmatrizen geeignet.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
descending_order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[descending_order]
eigenvectors = eigenvectors[:, descending_order]

explained_variance_ratio = eigenvalues / eigenvalues.sum()
first_two_components = eigenvectors[:, :2]
X_projected = X_standardized @ first_two_components

# Rückprojektion aus zwei Komponenten rekonstruiert nur den Anteil im
# zweidimensionalen PCA-Unterraum.
X_reconstructed_standardized = X_projected @ first_two_components.T
reconstruction_mse = float(
    np.mean((X_standardized - X_reconstructed_standardized) ** 2)
)

print("Erklärte Varianzanteile:", np.round(explained_variance_ratio, 4))
print("Kumuliert für zwei Komponenten:", round(explained_variance_ratio[:2].sum(), 4))
print("Rekonstruktions-MSE:", round(reconstruction_mse, 4))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    X_projected[:, 0],
    X_projected[:, 1],
    c=assignments,
    alpha=0.75,
)
ax.set_title("Cluster in der zweidimensionalen PCA-Projektion")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 5

PCA findet Richtungen maximaler Varianz, nicht automatisch die für Cluster oder Zielwerte wichtigsten Richtungen. Da die zusätzlichen Merkmale stark korreliert sind, können die ersten zwei Komponenten einen großen Teil der Gesamtvarianz erklären. Der Rekonstruktionsfehler misst den Informationsverlust in standardisierten Einheiten. Eine klare Projektion ist hilfreich, aber keine Garantie, dass die ursprüngliche Struktur vollständig erhalten bleibt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.